# SepsisGuard-UAR — Run the pipeline

Choose your available **A100 / high-RAM runtime** (CPU-only also works), then select **Runtime → Run all**. Approve the private Google Drive mount when prompted. Keep this notebook open until the final result appears.

This single workflow downloads/restores PhysioNet 2019 data, validates the cohort, produces EDA, locks patient partitions, runs leakage tests, fits causal train-only preprocessing, checks the PyTorch Dataset, and saves a verified private checkpoint. It automatically imports your completed data milestone when available and resumes a matching pipeline checkpoint on later runs.

All reports, fitted state, logs and the final `PIPELINE_RESULT.json` are kept together under `MyDrive/SepsisGuard-UAR/runs/<run-id>`. Raw files stay on the runtime disk; reusable data archives stay in Drive. If interrupted, reconnect a runtime and run this notebook again.

The implemented pipeline prepares research data. Predictive model training, calibration and retrieval experiments are additional components, and this notebook does not claim those are complete.

The allocated GPU model, VRAM and system RAM are recorded for each run. Data preparation executes on the CPU with either runtime type. GPU model training is a future component of this same workflow.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, uuid
from google.colab import drive

REPO_URL = 'https://github.com/shakiful/Sepsisguard-Uar.git'
PROJECT = Path('/content/sepsisguard-uar-unified')
# Keep previous notebooks and downloaded records intact.
if PROJECT.exists():
    assert (PROJECT/'.git').is_dir(), 'The chosen source folder is not a Git checkout.'
    remote = subprocess.check_output(['git','-C',str(PROJECT),'remote','get-url','origin'],text=True).strip()
    assert remote.lower().rstrip('/').removesuffix('.git') == REPO_URL.lower().removesuffix('.git'), 'Unexpected source repository.'
    dirty = subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain','--untracked-files=no'],text=True).strip()
    if dirty:
        PROJECT = Path('/content/sepsisguard-uar-unified-'+uuid.uuid4().hex[:8])
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch','main','--single-branch',REPO_URL,str(PROJECT)],check=True)
else:
    subprocess.run(['git','-C',str(PROJECT),'fetch','origin','main'],check=True)
    subprocess.run(['git','-C',str(PROJECT),'merge','--ff-only','origin/main'],check=True)
os.chdir(PROJECT)
compute = json.loads(subprocess.check_output([sys.executable,'scripts/compute_check.py'],text=True))
print(json.dumps(compute,indent=2))
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/SepsisGuard-UAR')
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev,download,torch]'],check=True)
old_data = Path('/content/sepsisguard-uar/data/raw')
DATA_ROOT = old_data if old_data.is_dir() else PROJECT/'data/raw'
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('Persistent project folder:',DRIVE_ROOT)


In [ ]:
# Leave automatic resume enabled for normal runs.
RUN_ID = 'auto'
command = [sys.executable,'-u','-m','scripts.run_pipeline',
           '--data-root',str(DATA_ROOT),'--storage-root',str(DRIVE_ROOT),'--run-id',RUN_ID]
process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:
    print(line,end='',flush=True)
if process.wait():
    raise RuntimeError('Pipeline stopped. Review the error above; completed Drive checkpoints remain available.')
print('Pipeline completed. Its final output identifies the saved run and result file.')
